# Sepsis (Camargo) — Old vs Improved

Single-source comparison: runs Camargo-style suffix sampling for both checkpoints (skipping any side that
already has chunked outputs), evaluates an **activity-only** metric set (Camargo doesn't predict time),
then plots them side by side.

**Pair:** `camargo/sepsis` &nbsp;·&nbsp; **Activity key:** `concept:name` &nbsp;·&nbsp; **Samples/case:** `100`

> Sampling uses ``ensure_sampled_camargo`` from ``_shared/camargo_sampling.py`` —
> argmax for ``mean_prediction`` and softmax-temperature draws for the
> ``predicted_suffixes``. Side-channel features and time are carried forward
> from the last observed event during autoregressive decoding (Camargo paper
> convention); the metric set drops every time metric since the model has no
> time head.

## How to run

1. Confirm the paths in the parameter cell point at the right Camargo checkpoints + test pickles.
2. Execute top-to-bottom. Sampling is skipped automatically if `SAMPLED_DIR_*` already contains `results_part_*.pkl`.
3. The bottom cells (table + overlay plot) tolerate either side being `None` — useful while one side is still being trained.


## Parameters

In [ ]:
# === PARAMETER CELL ===
# Tweak paths and knobs here. Everything below should run unchanged.
from pathlib import Path

PAIR_DIR = Path('.').resolve()   # the dir containing this notebook

# --- model checkpoints (None = side not yet trained / placeholder) ---
MODEL_OLD_PATH      = PAIR_DIR / 'old/Training/pkl/Sepsis_camargo_sharedcat_ngram5.pkl'
MODEL_IMPROVED_PATH = PAIR_DIR / 'improved/Training/pkl/Sepsis_camargo_sharedcat_allfeat_ngram5.pkl'

# --- encoded test pickles (must match what each checkpoint was trained on) ---
TEST_PKL_OLD      = PAIR_DIR / 'old/Loader/pkl/Sepsis_all_5_roles_test.pkl'
TEST_PKL_IMPROVED = PAIR_DIR / 'improved/Loader/pkl/Sepsis_all_5_allfeat_test.pkl'

# --- where the chunked sampling results land (also where batch_evaluate reads from) ---
SAMPLED_DIR_OLD      = PAIR_DIR / 'evaluation_results/old'
SAMPLED_DIR_IMPROVED = PAIR_DIR / 'evaluation_results/improved'

# --- comparison output ---
COMPARISON_PKL = PAIR_DIR / 'sepsis_old_vs_improved.pkl'
CAPTION        = 'Sepsis (Camargo)'

# --- Camargo model specs (cat/num feature indices into the test pickle, +
#     model class + ngram window). These mirror the camargo block in
#     src/interpretability/config/sepsis_config.py. ---
SPEC_OLD = dict(
    model_class='SharedCat_LSTM',
    cat_indices=(0, 1),                    # concept:name, org:group
    num_indices=(0,),                      # case_elapsed_time
    activity_feature='concept:name',
    ngram_size=5,
)
SPEC_IMPROVED = dict(
    model_class='SharedCat_LSTM',
    cat_indices=tuple(range(26)),          # all categoricals
    num_indices=tuple(range(8)),           # all numericals
    activity_feature='concept:name',
    ngram_size=5,
)

# --- sampling knobs (SamplingConfig kwargs) ---
CONCEPT_NAME        = 'concept:name'
ALL_CAT             = None
ALL_NUM             = None
GROWING_NUM_VALUES  = ['case_elapsed_time']
NUM_PROCESSES       = 16
SAMPLES_PER_CASE    = 100
SAVE_EVERY          = 50
RANDOM_ORDER        = False
USE_VARIANCE_CAT    = True
USE_VARIANCE_NUM    = True
SAMPLE_ARGMAX       = False
SAMPLING_TEMPERATURE = 1.0   # softmax temperature for stochastic decoding

# --- metric knobs (activity-only metric set; time metrics dropped for Camargo) ---
ACTIVITY_KEY      = 'concept:name'
EVENT_LABEL_LIST  = ['ER Registration', 'Leucocytes', 'CRP', 'LacticAcid', 'ER Triage', 'ER Sepsis Triage', 'IV Liquid', 'IV Antibiotics', 'Admission NC', 'Release A', 'Return ER', 'Admission IC', 'Release B', 'Release C', 'Release D', 'Release E']


## Setup

In [ ]:
import sys, importlib
from pathlib import Path

# Reach `src/` so `model.*`, `src.evaluation_metrics.*` resolve.
_REPO_ROOT = Path('.').resolve()
while not (_REPO_ROOT / 'src').is_dir() and _REPO_ROOT != _REPO_ROOT.parent:
    _REPO_ROOT = _REPO_ROOT.parent
for p in (
    str(_REPO_ROOT),
    str(_REPO_ROOT / 'src'),
    str(_REPO_ROOT / 'src' / 'reimplemented_comparable_approaches' / 'camargo_LSTM_suffix_pred'),
):
    if p not in sys.path:
        sys.path.insert(0, p)

# Reach the _shared helper module.
_SHARED = _REPO_ROOT / 'src' / 'interpretability' / 'improved_pipeline' / '_shared'
if str(_SHARED) not in sys.path:
    sys.path.insert(0, str(_SHARED))

import comparison_helpers
importlib.reload(comparison_helpers)
from comparison_helpers import (
    SamplingConfig, evaluate_dir, comparison_table, plot_overlay, save_results,
)

import camargo_sampling
importlib.reload(camargo_sampling)
from camargo_sampling import (
    ensure_sampled_camargo, default_metric_set_camargo, CamargoSpec,
)


## MC suffix sampling (skipped per side if chunks already exist)

In [ ]:
sampling_cfg = SamplingConfig(
    concept_name=CONCEPT_NAME,
    growing_num_values=GROWING_NUM_VALUES,
    all_cat=ALL_CAT,
    all_num=ALL_NUM,
    num_processes=NUM_PROCESSES,
    samples_per_case=SAMPLES_PER_CASE,
    sample_argmax=SAMPLE_ARGMAX,
    use_variance_cat=USE_VARIANCE_CAT,
    use_variance_num=USE_VARIANCE_NUM,
    random_order=RANDOM_ORDER,
    save_every=SAVE_EVERY,
)

if MODEL_OLD_PATH and TEST_PKL_OLD:
    ensure_sampled_camargo(
        MODEL_OLD_PATH, TEST_PKL_OLD, SAMPLED_DIR_OLD, sampling_cfg,
        spec=CamargoSpec(**SPEC_OLD),
        sampling_temperature=SAMPLING_TEMPERATURE,
    )
else:
    print('[skip] OLD side: missing MODEL_OLD_PATH or TEST_PKL_OLD')

if MODEL_IMPROVED_PATH and TEST_PKL_IMPROVED:
    ensure_sampled_camargo(
        MODEL_IMPROVED_PATH, TEST_PKL_IMPROVED, SAMPLED_DIR_IMPROVED, sampling_cfg,
        spec=CamargoSpec(**SPEC_IMPROVED),
        sampling_temperature=SAMPLING_TEMPERATURE,
    )
else:
    print('[skip] IMPROVED side: missing MODEL_IMPROVED_PATH or TEST_PKL_IMPROVED')


## Build metric set

In [ ]:
metrics = default_metric_set_camargo(
    activity_key=ACTIVITY_KEY,
    event_label_list=EVENT_LABEL_LIST,
)
print(f'metric set has {len(metrics)} entries (activity-only — Camargo has no time head)')


## Evaluate sampled outputs

In [ ]:
res_old, counts_old = (None, None)
res_improved, counts_improved = (None, None)

if SAMPLED_DIR_OLD.is_dir() and any(SAMPLED_DIR_OLD.glob('results_part_*.pkl')):
    res_old, counts_old = evaluate_dir(SAMPLED_DIR_OLD, metrics)
else:
    print('[skip] OLD eval: no chunks under', SAMPLED_DIR_OLD)

if SAMPLED_DIR_IMPROVED.is_dir() and any(SAMPLED_DIR_IMPROVED.glob('results_part_*.pkl')):
    res_improved, counts_improved = evaluate_dir(SAMPLED_DIR_IMPROVED, metrics)
else:
    print('[skip] IMPROVED eval: no chunks under', SAMPLED_DIR_IMPROVED)


## Side-by-side metric table

In [ ]:
import pandas as pd
df = comparison_table(res_old, res_improved)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
df


## Overlay plots

In [ ]:
plot_overlay(res_old, res_improved, counts_old, counts_improved, caption=CAPTION, pgf=False)


## Save comparison pickle

In [ ]:
save_results(
    COMPARISON_PKL,
    res_old=res_old, counts_old=counts_old,
    res_improved=res_improved, counts_improved=counts_improved,
    config_old=sampling_cfg, config_improved=sampling_cfg,
)


## Notes

- The comparison pickle written in the last cell holds both `(res_raw, counts)` pairs and the sampling configs that produced them. Re-load it later with `comparison_helpers.load_results(path)`.
- To force re-sampling, pass `force=True` into the explicit `ensure_sampled` calls (or just delete the chunked output dir).
- For dataset-specific metric tweaks, copy `default_metric_set` into a cell and edit it; the rest of the pipeline only cares that `metrics` is a `dict[str, metric]`.
